# 15 - The constants at -20 C

**Purpose.** Run `protocols/07-cold-constants.md` and publish `g(gain)`, `R(gain)` and the
pedestal at **-20 C**, at gains 50 and 200, so that session 06's sky frames can be turned into
electrons with constants measured where those frames were actually taken. It also measures the
same three quantities at -10 C in the same sitting, which gives the **temperature coefficient**
of each and a repeat of session 02 five weeks on.

**Why it exists.** Session 06 was shot at a -20 C setpoint. Everything in `results/` was measured
at -10 C. Using one on the other is a silent substitution, and this repo forbids those by name.
This is a repair session, and keeping it small is the point: two gains, no gain law, no
linearity, no dark current.

**What it is not for.** The gain law - two points cannot fit one and session 06 leans on no
interpolated gain, because 50 and 200 are both session 02 swept points. `ceiling(gain)` and full
well, which session 05 published and no sky frame approaches except in star cores. Dark current,
which session 03 bounded at -10 C and which is *smaller* at -20 C, so the bound holds a fortiori.
HCG, PRNU and the FPN test, all settled and none of them temperature-critical to the arithmetic
session 06 does.

**It does not move the project setpoint.** MISSION fixes cooling at -10 C. This characterises a
temperature the camera was accidentally run at; it does not adopt it.

**Three halves, run at different times.** Section 1 configures the bench and passes the two
blocking gates, section 2 captures the three arms, section 3 reads the frames back off disk and
publishes. Section 3 needs no camera and no kernel state from above it, so a wrong analysis costs
an evening and not a bench session.

## 1. Pre-flight

The panel is **not reconfigured**. Session 05 solved the patch colour that balances the four CFA
planes at gains 50 and 200 and published it; re-solving it here would put a second difference
inside a comparison whose whole purpose is to isolate temperature.

What is measured per arm, and never carried across one, is `t_sat`. It is cheap, and three
readings of it at the same colour are a free check on the backlight.

In [ ]:
import json
import pathlib
import sys
import time
import urllib.request

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session07"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

FULL_SCALE = 4095                    # ADC counts; the units rule in CLAUDE.md
GAINS = [50, 200]                    # exactly the two gains session 06 shot
ROI = (1408, 568, 1024, 1024)        # even origin and extent, or the Bayer phase shifts (L05)
OFFSET = 15                          # project_offset, fixed by session 01
PATCH_SERVER = "http://127.0.0.1:8765"

# The three arms, in the order they run.  Bracketed and not interleaved: a TEC
# needs minutes to move 10 C, so a frame-by-frame rotation would spend the
# session in transit.  What bracketing buys is that any drift across the evening
# shows up as a disagreement between a1 and a3 rather than as a temperature
# coefficient.
ARMS = [("a1", -10.0), ("a2", -20.0), ("a3", -10.0)]
COLD_ARM, WARM_ARMS = "a2", ("a1", "a3")

# Session 02's ladder, unchanged and for its reason (L10): with a linearly
# spaced ladder every point sits in the bright end, the low end is
# unconstrained, and that is where the slope absorbs its error.
RUNGS = [0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.8, 11.4, 19.2, 32.0, 54.0, 90.0]
N_LADDER = 4                         # two disjoint pairs per rung, so the variance has a repeat
N_BIAS = 20                          # per arm per gain: enough to assign the offset state
DISCARD = 2                          # frames dropped after a gain change (protocol)
DISCARD_EXPOSURE = 1                 # after an exposure change
FRAME_GAP_S = 0.2                    # readout, USB and the file write are what heat the sensor
HOLD_TIMEOUT_S = 300.0               # a hold that never ends is a cooler fault, not a wait
MAX_RETAKES = 10                     # per frame slot; more than this is not a transient
BIAS_LEVEL = 0                       # the bias block shoots with the panel driven black
TARGET_TSAT_S = 22.0                 # session 05's, so the rungs land where they did there

# Neither gain needs session 02's top-rung cap.  That cap bound at gains 300 and
# 450, where a frame's own sigma stops clearing the top code; at 50 and 200 the
# margin at 90% of t_sat is 5-20 sigma.
CLIP_FRAC_MAX = 1e-4

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
MIN_EXPOSURE = _bias["bias_exposure"]["value"]
HCG = _bias["hcg_threshold_gain"]["value"]

_lin = json.loads((RESULTS / "linearity_constants.json").read_text())
PATCH_OF_GAIN = {int(g): tuple(v) for g, v in _lin["patch_colour_per_gain"]["value"].items()}
TSAT_SESSION05 = {int(g): v for g, v in _lin["t_sat_per_gain"]["value"].items()}

_ptc = json.loads((RESULTS / "ptc_constants.json").read_text())
G_SESSION02 = {int(k): v for k, v in _ptc["system_gain"]["value"].items()}
G_ERR_SESSION02 = {int(k): v for k, v in _ptc["system_gain"]["uncertainty"].items()}

missing = [g for g in GAINS if g not in PATCH_OF_GAIN or g not in G_SESSION02]
assert not missing, (f"gains {missing} have no session 05 patch colour or no session 02 g; "
                     "this session compares row-for-row and cannot interpolate either")

RUNGS_CSV = RESULTS / "cold_rungs.csv"
BIAS_CSV = RESULTS / "cold_bias.csv"
CONSTANTS = RESULTS / "cold_constants.json"

print(f"{'gain':>5} {'patch (r,g,b)':>15} {'t_sat sess05':>13}")
for g in GAINS:
    print(f"{g:5d} {str(PATCH_OF_GAIN[g]):>15} {TSAT_SESSION05[g]:12.3f}s")
planned = len(ARMS) * len(GAINS) * (len(RUNGS) * N_LADDER + N_BIAS)
print(f"\n{planned} frames planned, {planned * ROI[2] * ROI[3] * 2 / 1e9:.2f} GB on C:")
print(f"{len(list(FRAMES.glob('*.fits')))} already on disk")

### The panel, and the one thing that can silently ruin a flat

The iPad's screen sleeps, the wake lock does not exist over plain `http://`, and a slept screen
is a **black panel** - which makes a flat frame a dark frame wearing a flat's header, with
nothing in the pixels to say so. The page reports a running redraw *count*, and only a moving
count proves the panel is being painted: iOS keeps the reporting timer alive on a screen it has
switched off while freezing `requestAnimationFrame`.

In [ ]:
def panel(path):
    with urllib.request.urlopen(f"{PATCH_SERVER}{path}", timeout=5) as r:
        return json.load(r)


def panel_frames():
    """The page's running redraw count, or None if it is not reporting one."""
    rep = panel("/level").get("refresh")
    return None if rep is None else rep["frames"]


def set_patch(rgb, settle_s=1.0):
    """Drive the panel and wait until the page says it has painted that colour.

    `/applied` is a handshake, not a measurement: the page polls every 300 ms
    and paints on its next redraw, so a capture landing in that gap gets the
    *previous* colour with nothing in the data to show it.  Waiting for
    `applied.seq` to reach the seq our own `/set` returned closes it.
    """
    want = panel(f"/set?rgb={rgb[0]},{rgb[1]},{rgb[2]}"
                 if isinstance(rgb, (tuple, list)) else f"/set?level={rgb}")["seq"]
    deadline = time.monotonic() + 60.0
    while time.monotonic() < deadline:
        applied = panel("/level").get("applied") or {}
        if applied.get("seq", -1) >= want:
            time.sleep(settle_s)
            return want
        time.sleep(0.3)
    raise TimeoutError(f"the panel never acknowledged seq {want}; is the page open and awake?")


def assert_panel_painting(window_s=3.0):
    """A moving redraw count, or the flats below are darks (light-source.md)."""
    first = panel_frames()
    assert first is not None, "the page is not reporting a redraw count -- reload it"
    time.sleep(window_s)
    second = panel_frames()
    assert second is not None and second > first, (
        f"redraw count stuck at {first}: the screen is asleep.  Auto-Lock must be Never, "
        "and the panel must be woken before anything below is trusted")
    return second - first


print(f"panel painting: {assert_panel_painting()} redraws in 3 s")

### Gate 1 - white balance, verified from the pixels (L01)

`asi.neutralise_white_balance` runs on open and reading the control back only proves the *control*
took. The evidence is in the pixels: the modal step between adjacent distinct values must be 16 on
all four planes. Greens at 16 with red at 17/18 and blue at 24 is the fingerprint of white balance
still being applied, and it inflated read noise by ~17% at every gain in a retired attempt.

**Nothing captured before this passes is usable.**

In [ ]:
existing = list(FRAMES.glob("*.fits"))
assert not existing, (f"{len(existing)} frames already in {FRAMES} -- delete the directory "
                      "to restart, or skip to section 3")

rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=GAINS[0], offset=OFFSET)
BIAS_EXPOSURE = rig.min_exposure_s()          # measured, never assumed

set_patch(BIAS_LEVEL)
for _ in range(DISCARD):
    asi.capture(rig, BIAS_EXPOSURE)

gate1 = {}
for _ in range(5):
    mosaic, _ = asi.capture(rig, BIAS_EXPOSURE)
    for name, plane in spatial.split(mosaic).items():   # stored values: the grid is 16 there
        gate1.setdefault(name, []).append(stats.value_step(plane))

for name in spatial.PLANES:
    print(f"  {name:2s} modal step {gate1[name]}")
bad = {n: s for n, s in gate1.items() if set(s) != {16}}
assert not bad, (f"white balance is still being applied: {bad} -- stop the session, "
                 "nothing captured from here is usable (L01)")
print(f"\ngate 1 passed on five frames.  bias exposure {BIAS_EXPOSURE * 1e6:.0f} us")

### Gate 2 - can this rig reach and hold -20 C?

**The gate that can end the session, and it has never been run on this rig.** Every bench setpoint
in this repo has been -10 C. -20 C is 10 C further down, indoors, in a warm room; session 06 held
it under a September night sky, which is not the same test.

The rule is `asi.cool_to`'s: in band (+/-0.5 C) for a continuous 30 s. What this cell adds is the
**duty cycle at the setpoint**, which is the headroom question. A TEC that settles at 95% duty has
settled, and will still lose the band when the sensor starts self-heating under an hour of
readout.

This runs the -20 C leg **first**, out of order, on purpose: if it fails, nothing else has been
spent. The arms then run in their proper order below, starting from a rig already known to be able
to get there.

In [ ]:
def cool(setpoint, tag):
    """Cool to a setpoint and keep the trace.  The reading only exists while we
    are the ones cooling: switch the cooler off and `Temperature` returns a flat
    zero (L03), taking the answer with it."""
    path = DATA / f"cooldown_{tag}.csv"
    with open(path, "w", newline="") as fh:
        fh.write("elapsed_s,temp_C,duty_pct\n")

        def show(elapsed, temp, duty):
            fh.write(f"{elapsed},{temp},{duty}\n")
            fh.flush()                       # the reading exists nowhere else
            if int(elapsed) % 30 == 0:
                print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)

        trace = asi.cool_to(rig, setpoint, log=show)

    temps = [t for _, t, _ in trace if t is not None]
    print(f"  settled at {setpoint} C in {trace[-1][0]:.0f} s; "
          f"{temps[0]} -> {temps[-1]} C, duty {trace[-1][2]}%")
    return trace


DUTY_HEADROOM_MAX = 90               # settling above this leaves nothing for self-heating

probe = cool(-20.0, "gate2_probe")
duty_at_20 = probe[-1][2]
assert duty_at_20 <= DUTY_HEADROOM_MAX, (
    f"settled at -20 C on {duty_at_20}% duty, over the {DUTY_HEADROOM_MAX}% bar.  There is no "
    "headroom for the self-heating an hour of readout causes, so the arm would lose the band "
    "part-way through.  Gate 2's fallback applies: stop, and publish session 06 against the "
    "-10 C constants with the coefficient named as unmeasured (protocols/07-cold-constants.md)")
print(f"\ngate 2 passed: -20 C reachable at {duty_at_20}% duty, "
      f"{DUTY_HEADROOM_MAX - duty_at_20} points of headroom")

## 2. The session - three arms

```
arm 1: -10 C      arm 2: -20 C      arm 3: -10 C
```

Per arm, per gain: gate 3 measures `t_sat` cold at that arm's own temperature, then the twelve
rungs, then that arm-and-gain's own bias block shot adjacent to it. The bias block is **not** a
re-measurement of read noise for its own sake - it is the pedestal this arm's signal is measured
against, taken minutes rather than hours away. L14's cautionary tale is a dark sitting one count
below a bias shot four hours earlier, which produced a negative dark current.

Twenty bias frames rather than session 02's ten, because the offset state has to be *assigned*
across them (`stats.offset_state`) rather than assumed, and a ten-frame group is thin for that.

In [ ]:
def plane_means(mosaic):
    return {k: float(stats.to_adc(v).mean()) for k, v in spatial.split(mosaic).items()}


def hold_for_temperature(where, exposure_s, setpoint):
    """Shoot discards until the sensor has been in band for `asi.RECOVER_S`.

    Discards are shot at the run's own exposure, so the duty cycle during the
    hold is the duty cycle being recovered from.
    """
    t0, in_band_since, worst = time.monotonic(), None, None
    while True:
        _, header = asi.capture(rig, exposure_s, imagetyp="FLAT")    # discarded on purpose
        time.sleep(FRAME_GAP_S)
        temp, now = header["CCD-TEMP"], time.monotonic()
        worst = temp if worst is None or (temp is not None and temp > worst) else worst

        if temp is not None and abs(temp - setpoint) <= asi.BAND_C:
            in_band_since = now if in_band_since is None else in_band_since
            if now - in_band_since >= asi.RECOVER_S:
                held = now - t0
                print(f"      held {held:.0f} s at {where}, peak {worst} C, back at {temp} C",
                      flush=True)
                return held
        else:
            in_band_since = None

        if now - t0 > HOLD_TIMEOUT_S:
            raise TimeoutError(
                f"{HOLD_TIMEOUT_S:.0f} s of holding at {where} and still {temp} C: the cooler "
                "is not keeping up.  Stop the session and check the ambient and the fan -- do "
                "not widen the band")


def capture_frames(n, exposure_s, imagetyp, name, where, setpoint):
    """`n` in-band frames at one setting, written as `name`_000.fits onwards."""
    retaken, held_s = 0, 0.0
    for i in range(n):
        for _ in range(MAX_RETAKES + 1):
            mosaic, header = asi.capture(rig, exposure_s, imagetyp=imagetyp)
            temp = header["CCD-TEMP"]
            if temp is not None and abs(temp - setpoint) <= asi.BAND_C:
                break
            retaken += 1
            print(f"    ! {where} frame {i} at {temp} C, retaking", flush=True)
            if temp is not None and temp > setpoint + asi.BAND_C:
                held_s += hold_for_temperature(where, exposure_s, setpoint)
            else:
                time.sleep(FRAME_GAP_S)
        else:
            raise RuntimeError(
                f"{MAX_RETAKES} retakes at {where} and still {temp} C.  This is no longer a "
                "transient -- stop the session rather than filling the curve with frames "
                "nobody can defend")

        F.write(FRAMES / f"{name}_{i:03d}.fits", mosaic, header)
        time.sleep(FRAME_GAP_S)
    return retaken, held_s


def measure_tsat(gain, guess_s, setpoint, tries=5):
    """Gate 3: `t_sat` at this arm's own temperature, from the measured flux.

    Returns `(t_sat, flux, plane, probe_exptime, converged)`.  The pedestal it
    subtracts is session 01's fitted law -- a *prediction* used to place a rung,
    never an input to a result.  Every signal in section 3 is measured against
    this arm's own bias block instead.
    """
    branch = _bias["pedestal_fit"]["value"]["hcg" if gain >= HCG else "lcg"]
    ped = branch["A"] + branch["B"] * 10.0 ** (gain / 200.0)
    head, e = FULL_SCALE - ped, guess_s

    asi.configure(rig, gain=gain, offset=OFFSET)
    for _ in range(DISCARD):
        asi.capture(rig, guess_s, imagetyp="FLAT")

    for _ in range(tries):
        asi.capture(rig, e, imagetyp="FLAT")              # the discard after the change
        mosaic, h = asi.capture(rig, e, imagetyp="FLAT")
        means = plane_means(mosaic)
        plane = max(means, key=means.get)
        signal = means[plane] - ped
        if 0.2 * head <= signal <= 0.7 * head:
            flux = signal / h["EXPTIME"]
            return head / flux, flux, plane, h["EXPTIME"], True
        e = min(max(e * 0.45 * head / max(signal, 1.0), MIN_EXPOSURE), 60.0)
    flux = signal / h["EXPTIME"]
    return head / flux, flux, plane, h["EXPTIME"], False

### The arm loop

Each arm cools, then shoots both gains. `t_sat` is re-measured inside every arm and never carried
across one: three readings at the same patch colour are also a free check on the backlight, and a
drift in them shows up here rather than inside a gain.

In [ ]:
gate3, arm_log = [], []
t0_session = time.monotonic()

for arm, setpoint in ARMS:
    print(f"\n=== arm {arm} at {setpoint} C " + "=" * 40, flush=True)
    cool(setpoint, arm)
    assert_panel_painting()

    for gain in GAINS:
        set_patch(PATCH_OF_GAIN[gain])
        t_sat, flux, plane, probe_s, ok = measure_tsat(
            gain, max(0.4 * TSAT_SESSION05[gain], MIN_EXPOSURE), setpoint)
        gate3.append({"arm": arm, "setpoint_c": setpoint, "gain": gain, "plane": plane,
                      "probe_s": probe_s, "flux": flux, "t_sat_s": t_sat,
                      "t_sat_sess05": TSAT_SESSION05[gain], "converged": ok})
        print(f"  gain {gain}: t_sat {t_sat:.3f}s on {plane} "
              f"({t_sat / TSAT_SESSION05[gain]:.3f}x session 05)"
              f"{'' if ok else '   <- probe did not converge'}", flush=True)

        ladder = [r / 100.0 * t_sat for r in RUNGS]
        assert min(ladder) >= MIN_EXPOSURE, (
            f"faintest rung {min(ladder) * 1e6:.0f} us is under the shutter floor -- the bench "
            "is too bright, and no ladder fixes that")

        t_gain, retaken, held = time.monotonic(), 0, 0.0
        for _ in range(DISCARD):
            asi.capture(rig, ladder[0], imagetyp="FLAT")
        for ri, exposure_s in enumerate(ladder):
            for _ in range(DISCARD_EXPOSURE):
                asi.capture(rig, exposure_s, imagetyp="FLAT")
            r, h = capture_frames(N_LADDER, exposure_s, "FLAT",
                                  f"flat_{arm}_g{gain:03d}_r{ri:02d}",
                                  f"{arm} gain {gain} rung {ri}", setpoint)
            retaken, held = retaken + r, held + h
            print(f"    rung {ri:>2}/{len(RUNGS)}  {exposure_s * 1e3:9.3f} ms  "
                  f"{(time.monotonic() - t_gain) / 60:5.1f} min", flush=True)

        set_patch(BIAS_LEVEL)
        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, BIAS_EXPOSURE, imagetyp="BIAS")
        r, h = capture_frames(N_BIAS, BIAS_EXPOSURE, "BIAS", f"bias_{arm}_g{gain:03d}",
                              f"{arm} gain {gain} pedestal", setpoint)
        retaken, held = retaken + r, held + h

        arm_log.append({"arm": arm, "gain": gain, "retaken": retaken, "held_s": held,
                        "minutes": (time.monotonic() - t_gain) / 60,
                        "duty_end_pct": rig.get("CoolPowerPerc")})
        print(f"  gain {gain} done: {retaken} retaken, {held / 60:.1f} min held, "
              f"duty {rig.get('CoolPowerPerc')}%", flush=True)

pd.DataFrame(gate3).to_csv(DATA / "gate3.csv", index=False)
pd.DataFrame(arm_log).to_csv(DATA / "arms.csv", index=False)
print(f"\n{(time.monotonic() - t0_session) / 60:.0f} min in all; "
      f"{len(list(FRAMES.glob('*.fits')))} frames written")
print(f"gate 3 and the arm log written to {DATA} (data/, not results/)")

### Closing down

Cooler off, camera released. Closing drops the cooler, and `Gain` and `Offset` survive it while
the TEC does not - so nothing below may assume the rig is still cold.

In [ ]:
rig.set("CoolerOn", 0, verify=False)
set_patch("free")                    # hand the panel back to whoever holds the iPad
rig.close()
print("cooler off, camera closed, panel released")

## 3. The analysis

**Everything from here reads disk and nothing else.** The frames, `data/session07/gate3.csv`,
`data/session07/arms.csv` and session 02's `results/ptc_constants.json` are the whole input, so
the analysis can be re-run and corrected without costing a bench session.

It runs `protocols/07-cold-constants.md`'s eight rules, fixed before the data existed:

| rule | what it does | published as |
|---|---|---|
| 1 | pedestal from this arm-and-gain's own bias block, near state only | `pedestal` in `cold_bias.csv` |
| 2 | signal per plane, never on the frame mean | `signal` in `cold_rungs.csv` |
| 3 | variance from a pair difference, blind to fixed pattern | `var_pair` |
| 4 | `g` as a slope with `R` passed in, not as the intercept | `system_gain_cold` |
| 5 | `R` in counts from the bias block, near state only | `read_noise_cold` |
| 6 | the coefficient is -20 C against the *mean* of the two -10 C arms | `temperature_coefficient` |
| 7 | session 02 recorded beside the new numbers, never averaged in | `vs_session02` |
| 8 | nothing here overwrites `ptc_constants.json` | a separate file |

In [ ]:
# Section 3 is self-contained on purpose: a fresh kernel can run from here down,
# because the frames and the three files below are the whole input.
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session07"
FRAMES = DATA / "frames"

FULL_SCALE = 4095
GAINS = [50, 200]
ARMS = [("a1", -10.0), ("a2", -20.0), ("a3", -10.0)]
COLD_ARM, WARM_ARMS = "a2", ("a1", "a3")
RUNGS = [0.3, 0.5, 0.85, 1.4, 2.4, 4.0, 6.8, 11.4, 19.2, 32.0, 54.0, 90.0]
CLIP_FRAC_MAX = 1e-4
PLANES = spatial.PLANES

RUNGS_CSV = RESULTS / "cold_rungs.csv"
BIAS_CSV = RESULTS / "cold_bias.csv"
CONSTANTS = RESULTS / "cold_constants.json"

_ptc = json.loads((RESULTS / "ptc_constants.json").read_text())
G_SESSION02 = {int(k): v for k, v in _ptc["system_gain"]["value"].items()}
G_ERR_SESSION02 = {int(k): v for k, v in _ptc["system_gain"]["uncertainty"].items()}
_gain_csv = pd.read_csv(RESULTS / "ptc_gain.csv")
R_SESSION02 = (_gain_csv[_gain_csv.gain.isin(GAINS)]
               .groupby("gain").R_counts.first().to_dict())

gate3 = pd.read_csv(DATA / "gate3.csv")
arms_log = pd.read_csv(DATA / "arms.csv")


def planes_adc(path):
    """One frame as four float64 CFA planes in ADC counts, plus its header."""
    mosaic, header = F.read(path)
    return spatial.split(stats.to_adc(mosaic).astype(np.float64)), header


def pair_variance(a, b):
    """Rule 3: the temporal variance, blind to fixed pattern by construction."""
    return float(np.var(a - b, ddof=1) / 2.0)


print(f"{len(list(FRAMES.glob('*.fits')))} frames on disk")
print(gate3.to_string(index=False))

### Rules 1 and 5 - the pedestal and `R`, from each arm's own bias block

The offset state is assigned across the twenty frames before either number is taken. A state hop
is a black-level move of about one count, not read noise: including one inflates `R` by the step
and drags the pedestal by a fraction of it. Session 03 found the hop on 4.1% of frames, which on
a twenty-frame block is about one.

Both numbers are per plane. `R` here is a *measurement this session makes*, not a value passed in
from session 01 - the read noise at -20 C is one of the three quantities the session exists to
publish - and it is what rule 4's fit then takes as its fixed intercept.

In [ ]:
bias_rows, pedestal, R_counts = [], {}, {}
for arm, setpoint in ARMS:
    for g in GAINS:
        files = sorted(FRAMES.glob(f"bias_{arm}_g{g:03d}_*.fits"))
        assert files, f"no bias frames for {arm} gain {g}"
        levels = {p: [] for p in PLANES}
        planes = []
        for f in files:
            pl, h = planes_adc(f)
            planes.append(pl)
            for p in PLANES:
                levels[p].append(float(pl[p].mean()))

        # The state is assigned on the frame level -- the mean over the four
        # planes -- because a black-level move is common to all of them.  Doing
        # it per plane would let noise in one plane relabel a frame.
        frame_level = np.mean([levels[p] for p in PLANES], axis=0)
        state = stats.offset_state(frame_level)
        near = ~state["far"]

        for p in PLANES:
            vals = np.asarray(levels[p])[near]
            keep = [pl[p] for pl, k in zip(planes, near) if k]
            # Pooled over pixels, not averaged: the mean of per-pixel standard
            # deviations is biased low, and R is a variance quantity.
            temporal = float(np.sqrt(np.mean(
                np.var(np.stack(keep, axis=0), axis=0, ddof=1))))
            bias_rows.append({
                "arm": arm, "setpoint_c": setpoint, "gain": g, "plane": p,
                "n_frames": len(files), "n_near": int(near.sum()),
                "pedestal": float(vals.mean()),
                "pedestal_sd": float(vals.std(ddof=1)),
                "R_counts": temporal,
                "state_separation": state["separation"],
                "state_far_frac": float((~near).mean()),
            })
        block = bias_rows[-len(PLANES):]
        pedestal[(arm, g)] = {r["plane"]: r["pedestal"] for r in block}
        R_counts[(arm, g)] = {r["plane"]: r["R_counts"] for r in block}

bias_tbl = pd.DataFrame(bias_rows)
bias_tbl.to_csv(BIAS_CSV, index=False)
print(f"wrote {BIAS_CSV}  ({len(bias_tbl)} rows)")
print()
print(bias_tbl.pivot_table(index=["arm", "gain"], columns="plane",
                           values=["pedestal", "R_counts"]).round(4).to_string())
far = bias_tbl[bias_tbl.state_far_frac > 0]
print(f"\noffset state: {len(far) // 4} of {len(bias_tbl) // 4} blocks had a frame in a far "
      f"state; those frames are excluded from both numbers above")

### Rules 2, 3 and 4 - the rung table, and `g` as a slope

Signal is the plane mean above this arm-and-gain's own pedestal. Variance is half the variance of
a frame-pair difference, which is blind to fixed pattern by construction - that is what makes the
single-frame column beside it a check rather than a restatement.

`g` is the **slope** of variance against signal with the intercept fixed at the `R` measured
above, never the ratio at one rung and never the free intercept (L10). The free-intercept fit is
kept beside it as a cross-check and is not the answer.

In [ ]:
t_sat = {(r.arm, r.gain): r.t_sat_s for r in gate3.itertuples()}

rows = []
for arm, setpoint in ARMS:
    for g in GAINS:
        for ri in range(len(RUNGS)):
            files = sorted(FRAMES.glob(f"flat_{arm}_g{g:03d}_r{ri:02d}_*.fits"))
            if not files:
                continue
            loaded = [planes_adc(f) for f in files]
            exptime = float(np.mean([h["EXPTIME"] for _, h in loaded]))
            temp = float(np.mean([h["CCD-TEMP"] for _, h in loaded]))
            for p in PLANES:
                a = [pl[p] for pl, _ in loaded]
                var_pairs = [pair_variance(a[0], a[1]), pair_variance(a[2], a[3])]
                rows.append({
                    "arm": arm, "setpoint_c": setpoint, "gain": g, "plane": p, "rung": ri,
                    "rung_pct_planned": RUNGS[ri],
                    "pct_of_tsat": 100 * exptime / t_sat[(arm, g)],
                    "exptime_s": exptime, "pedestal": pedestal[(arm, g)][p],
                    "signal": float(np.mean([x.mean() for x in a])) - pedestal[(arm, g)][p],
                    "var_pair": float(np.mean(var_pairs)),
                    "var_pair_spread": float(abs(var_pairs[0] - var_pairs[1])),
                    "var_single": float(np.mean([np.var(x, ddof=1) for x in a])),
                    "sat_frac": float(np.mean([(x >= FULL_SCALE).mean() for x in a])),
                    "R_counts": R_counts[(arm, g)][p], "ccd_temp": temp,
                    "n_frames": len(files),
                })
    print(f"  arm {arm} done", flush=True)

rungs = pd.DataFrame(rows)
rungs["usable"] = ((rungs.sat_frac < CLIP_FRAC_MAX) & (rungs.signal > 0)
                   & (rungs.var_pair > rungs.R_counts ** 2))
rungs.to_csv(RUNGS_CSV, index=False)
print(f"\nwrote {RUNGS_CSV}  ({len(rungs)} rows, {int(rungs.usable.sum())} usable)")

In [ ]:
def ptc_fit(signal, var, R2):
    """Rule 4.  The slope-fit g with the intercept fixed at R^2, the
    free-intercept fit beside it, and the scatter about the fit.

    Weights are 1/var^2: constant relative error, which is what a variance
    estimate actually has.  This is session 02's `ptc_fit`, unchanged -- the
    same arithmetic on both sides is what makes rule 7's comparison mean
    anything.
    """
    S, V = np.asarray(signal, float), np.asarray(var, float)
    w = 1.0 / V ** 2

    m = float(np.sum(w * S * (V - R2)) / np.sum(w * S * S))       # var - R^2 = S/g
    resid = (V - (m * S + R2)) / V
    dof = max(len(S) - 1, 1)
    se_m = float(np.sqrt(np.sum(w * (V - m * S - R2) ** 2) / dof / np.sum(w * S * S)))

    X = np.column_stack([S, np.ones_like(S)])                     # free intercept
    sw = np.sqrt(w)
    (a, b), *_ = np.linalg.lstsq(X * sw[:, None], V * sw, rcond=None)

    return {"g": 1.0 / m, "g_err": abs(se_m / m ** 2),
            "resid_pct": float(np.sqrt(np.mean(resid ** 2)) * 100),
            "g_free": 1.0 / float(a) if a > 0 else np.nan,
            "R_fit": float(np.sqrt(b)) if b > 0 else np.nan,
            "n_rungs": int(len(S)), "S_max": float(S.max())}


fits = []
for (arm, g, p), d in rungs[rungs.usable].groupby(["arm", "gain", "plane"]):
    d = d.sort_values("signal")
    row = ptc_fit(d.signal, d.var_pair, R_counts[(arm, g)][p] ** 2)
    row.update({"arm": arm, "gain": g, "plane": p,
                "setpoint_c": dict(ARMS)[arm],
                "R_counts": R_counts[(arm, g)][p],
                "R_e": row["g"] * R_counts[(arm, g)][p],
                "pedestal": pedestal[(arm, g)][p]})
    fits.append(row)

gain_tbl = pd.DataFrame(fits).sort_values(["arm", "gain", "plane"]).reset_index(drop=True)
per_arm = gain_tbl.groupby(["arm", "gain"]).agg(
    setpoint_c=("setpoint_c", "first"), g=("g", "mean"), g_sd=("g", "std"),
    g_err=("g_err", "mean"), resid_pct=("resid_pct", "mean"),
    R_counts=("R_counts", "mean"), R_e=("R_e", "mean"),
    pedestal=("pedestal", "mean"), n_rungs=("n_rungs", "min"))
per_arm["plane_spread_pct"] = 100 * per_arm.g_sd / per_arm.g

print("per arm and gain, mean over the four CFA planes:")
print(per_arm.round(4).to_string())
print("\nfitted intercept as R in counts, against the block measurement "
      "(a cross-check, never the answer -- L10):")
print(gain_tbl.pivot_table(index=["arm", "gain"], values=["R_fit", "R_counts"])
      .round(4).to_string())

### Rule 6 - the temperature coefficient, and what the two warm arms cost it

The -10 C figure is the **mean of arms 1 and 3**, and its uncertainty is **half their
difference** - not the scatter within either. That is the honest error bar: the two arms are
separated by the whole -20 C leg, so anything that drifted across the evening is inside that gap
and nothing inside one arm can see it.

**Read the `arm_disagreement` row first.** If the two warm arms differ by more than their own
repeat scatter, this session measured drift and not temperature, and every coefficient below is
a description of the bench.

In [ ]:
def arm_value(column):
    """One number per (arm, gain), as a frame indexed by gain."""
    return per_arm[column].unstack(0)


coef_rows = []
for column, unit in (("g", "e-/ADC count"), ("R_counts", "ADC counts"),
                     ("R_e", "e-"), ("pedestal", "ADC counts")):
    v = arm_value(column)
    warm = v[list(WARM_ARMS)].mean(axis=1)
    disagree = (v[WARM_ARMS[1]] - v[WARM_ARMS[0]]).abs()
    cold = v[COLD_ARM]
    for g in GAINS:
        coef_rows.append({
            "quantity": column, "unit": unit, "gain": g,
            "at_-20": float(cold[g]), "at_-10": float(warm[g]),
            "delta": float(cold[g] - warm[g]),
            "delta_pct": float(100 * (cold[g] / warm[g] - 1)),
            "arm_disagreement": float(disagree[g]),
            "arm_disagreement_pct": float(100 * disagree[g] / warm[g]),
            "resolved": bool(abs(cold[g] - warm[g]) > disagree[g]),
        })

coef = pd.DataFrame(coef_rows)
print(coef.round(5).to_string(index=False))
print()
for r in coef.itertuples():
    verdict = ("moved, and by more than the two warm arms disagree" if r.resolved else
               "inside the warm arms' own disagreement -- not resolved by this session")
    print(f"  {r.quantity:>9} at gain {r.gain:3d}: {r.delta:+.4f} {r.unit}  "
          f"({r.delta_pct:+.2f}%)  {verdict}")

### Rule 7 - session 02 beside the new numbers, never averaged into them

The two -10 C arms are a repeat of session 02 at gains 50 and 200, five weeks and one panel
reconfiguration later. Agreement is the thing that vouches for this bench; disagreement is a
finding about the bench and is published as one, **not averaged away**.

This is also the only check that separates "the constants moved with temperature" from "the
constants moved because something about the bench changed". Without it the -20 C numbers are
un-anchored.

In [ ]:
warm_g = arm_value("g")[list(WARM_ARMS)].mean(axis=1)
warm_R = arm_value("R_counts")[list(WARM_ARMS)].mean(axis=1)

vs02 = pd.DataFrame({
    "g_this_session_-10": warm_g,
    "g_session02": pd.Series(G_SESSION02).reindex(GAINS),
    "R_this_session_-10": warm_R,
    "R_session02": pd.Series(R_SESSION02).reindex(GAINS),
})
vs02["g_agreement_pct"] = 100 * (vs02["g_this_session_-10"] / vs02.g_session02 - 1)
vs02["R_agreement_pct"] = 100 * (vs02["R_this_session_-10"] / vs02.R_session02 - 1)
vs02["within_session02_err"] = (vs02.g_agreement_pct.abs() / 100 * vs02.g_session02
                                <= pd.Series(G_ERR_SESSION02).reindex(GAINS))

print(vs02.round(4).to_string())
print()
worst = float(vs02.g_agreement_pct.abs().max())
print(f"worst g disagreement against session 02: {worst:.2f}%")
print("  " + ("two sittings five weeks apart agree, which is stronger evidence than either "
               "alone; session 02 gains the repeat it did not have"
              if worst < 1.0 else
              "this is not a repeat.  Something changed between the sessions -- the panel, the "
              "room, the camera -- and finding it is the work.  Do NOT average the two"))

### Publishing

`cold_constants.json`, with provenance on every entry. **Nothing here touches
`ptc_constants.json`**: session 02's constants are correct at their own setpoint and stay exactly
as published. Session 06 reads this file instead, because its frames are at this temperature.

In [ ]:
on_disk = sorted(FRAMES.glob("*.fits"))
measured_on = str(F.read(on_disk[0])[1]["DATE-OBS"])[:10]
n_frames = len(on_disk)


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "15_cold_constants.ipynb", "note": note}


def nn(v):
    """None for anything that is not a finite number: a null is a published
    statement that the quantity was not measured, and NaN is not JSON."""
    return None if v is None or not np.isfinite(v) else round(float(v), 6)


def by_gain(series):
    return {int(g): nn(v) for g, v in series.items()}


def by_gain_plane(arm, column):
    d = gain_tbl[gain_tbl.arm == arm]
    return {int(g): {r.plane: nn(getattr(r, column)) for r in s.itertuples()}
            for g, s in d.groupby("gain")}


cold = per_arm.xs(COLD_ARM, level="arm")
warm = arm_value("g")[list(WARM_ARMS)].mean(axis=1)
warm_disagree = (arm_value("g")[WARM_ARMS[1]] - arm_value("g")[WARM_ARMS[0]]).abs()
duty_cold = arms_log[arms_log.arm == COLD_ARM].duty_end_pct

constants = {
    "setpoint": constant(
        -20.0, "C", 0.5,
        "the setpoint session 06 was actually shot at.  MISSION fixes the project at -10 C and "
        "this session does not change that: it characterises a temperature the camera was run "
        "at by accident, so that those frames can be published without a silent substitution"),
    "system_gain_cold": constant(
        by_gain(cold.g), "e- per ADC count", by_gain(cold.g_sd),
        "rule 4 of protocols/07-cold-constants.md: the slope of pair-difference variance against "
        "signal at -20 C, per CFA plane, with the intercept fixed at this session's own measured "
        "R and never taken from the fit (L10).  Uncertainty is the spread across the four planes; "
        "the formal fit error is system_gain_cold_fit_err.  Per-plane values are in "
        "system_gain_cold_per_plane and every rung is in cold_rungs.csv"),
    "system_gain_cold_per_plane": constant(
        by_gain_plane(COLD_ARM, "g"), "e- per ADC count", None,
        "system_gain_cold, per CFA plane.  MISSION's per-plane framing needs g per plane wherever "
        "a plane's own electrons are being counted, which is every F_sky number session 06 "
        "publishes"),
    "system_gain_cold_fit_err": constant(
        by_gain(cold.g_err), "e- per ADC count", None,
        "the weighted-fit standard error on the slope, averaged over the four planes -- the "
        "precision of one fit, as against the plane-to-plane spread published as the uncertainty"),
    "read_noise_cold": constant(
        by_gain(cold.R_counts), "ADC counts", None,
        "rule 5: the temporal standard deviation across this arm's 20-frame bias block, per "
        "plane, on the frames in the near offset state only.  A state hop is a black-level move "
        "of about one count, not read noise, and including one inflates R by the step.  This is "
        "measured here rather than passed in from session 01, because read noise at -20 C is one "
        "of the three quantities this session exists to publish"),
    "read_noise_cold_e": constant(
        by_gain(cold.R_e), "e-", None,
        "read_noise_cold x system_gain_cold, per plane, then averaged.  This is the R the model's "
        "R^2/t term consumes for session 06"),
    "pedestal_cold": constant(
        by_gain(cold.pedestal), "ADC counts at offset 15", None,
        "rule 1: the plane mean of this arm's own bias block, near offset state only.  It is the "
        "number F_sky is most exposed to -- at gain 50, 120 s of L32's green sky is about 35 "
        "counts, so a one-count pedestal error is a 3% error in the sky rate"),
    "temperature_coefficient": constant(
        {q: {int(r.gain): nn(r.delta) for r in coef[coef.quantity == q].itertuples()}
         for q in coef.quantity.unique()},
        "value at -20 C minus value at -10 C, in each quantity's own unit", None,
        "rule 6.  The -10 C figure is the mean of arms 1 and 3, which bracket the -20 C arm, and "
        "the uncertainty on the comparison is arm_disagreement -- half their difference -- not "
        "the scatter within either.  A coefficient smaller than that disagreement is published "
        "as unresolved, because the two are not distinguishable by this session"),
    "temperature_coefficient_resolved": constant(
        {q: {int(r.gain): bool(r.resolved)
             for r in coef[coef.quantity == q].itertuples()}
         for q in coef.quantity.unique()},
        "boolean per quantity per gain", None,
        "whether the -20 C to -10 C change exceeds the two warm arms' own disagreement.  False "
        "does not mean 'no change' -- it means this session cannot tell the change from its own "
        "drift, which is the honest statement and the one that bounds the substitution session "
        "06 would otherwise have made"),
    "arm_disagreement": constant(
        {q: {int(r.gain): nn(r.arm_disagreement)
             for r in coef[coef.quantity == q].itertuples()}
         for q in coef.quantity.unique()},
        "absolute difference between the two -10 C arms, each quantity's own unit", None,
        "the bench's own stability across the evening, measured rather than assumed.  Arms 1 and "
        "3 sit either side of the whole -20 C leg, so anything that drifted is inside this number "
        "and nothing inside a single arm can see it.  It is the error bar on every coefficient "
        "above"),
    "vs_session02": constant(
        {int(g): {"g_this": nn(warm[g]), "g_session02": nn(G_SESSION02[g]),
                  "agreement_pct": nn(100 * (warm[g] / G_SESSION02[g] - 1)),
                  "R_this": nn(warm_R[g]), "R_session02": nn(R_SESSION02.get(g))}
         for g in GAINS},
        "e- per ADC count, and % against session 02", None,
        "rule 7: this session's own -10 C arms against ptc_constants.json, five weeks and one "
        "panel reconfiguration later.  Recorded beside the new numbers and never averaged into "
        "them.  Agreement vouches for the bench and anchors the -20 C arm; disagreement is a "
        "finding about the bench, and the -20 C numbers are not publishable until it is "
        "explained"),
    "cooler_duty_at_cold": constant(
        nn(duty_cold.max()) if len(duty_cold) else None, "%", None,
        "gate 2: the worst TEC duty cycle at the end of a -20 C gain block.  The gate's bar is "
        "90%, and the reason is self-heating: a TEC that settles near its limit holds the band "
        "until an hour of readout arrives.  This is the number that says whether -20 C is a "
        "setpoint this rig can actually work at, as opposed to reach"),
    "t_sat_per_arm": constant(
        {a: {int(r.gain): nn(r.t_sat_s) for r in gate3[gate3.arm == a].itertuples()}
         for a in gate3.arm.unique()},
        "s", None,
        "gate 3: t_sat measured cold inside each arm at session 05's published patch colour, "
        "never carried across an arm.  Three readings at one colour are also a free check on the "
        "backlight -- a drift in these is a drift in the panel, and it shows up here rather than "
        "inside a gain"),
}

with open(CONSTANTS, "w", encoding="utf8") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS} with {len(constants)} constants, provenance on each")
print()
for k, v in constants.items():
    print(f"  {k:<32} {json.dumps(v['value'])[:70]}")
print()
print(f"{n_frames} frames, captured {measured_on}")
print(f"also written: {RUNGS_CSV.name}, {BIAS_CSV.name}")
print("\nptc_constants.json is untouched, deliberately: session 02 is correct at its own "
      "setpoint (rule 8).")